In [74]:
from functools import cache
from itertools import product
import numpy as np
import random
import cProfile
import pstats
import copy

K.<ω> = CyclotomicField(3)       
O = K.ring_of_integers()     

@cache 
def cubic_residue(a, π):
    a = O(a)
    red, w, e, one = residue_map(π)
    if red(a) == 0:
        return 0
    r = red(a)**e
    if r == one:
        return 0
    elif r == w:
        return 1
    else:
        return 2
@cache
def residue_map(π):
    π = O(π)
    P = O.fractional_ideal(π)
    k = K.residue_field(P) 
    red = k.reduction_map()
    w = red(ω)
    e = (π.norm() - 1) // 3
    one = k(1)
    return red, w, e, one

"""
Returns the base ω log of the general cubic reciprocity (a/b)
"""
@cache
def cr(a,b):
    res = 0
    for π, exp in prime_factors(b):
        local = cubic_residue(a, π)
        res += local*exp 
    return res % 3
    
"""
Returns the prime factorization of b in the ring O
"""
@cache
def prime_factors(b):
    return O(b).factor()

"""
Returns the prime above a 1 mod 3 prime in the ring O
"""
@cache
def prime_above(p):
    # p needs to be a prime 1 mod 3 here
    fac = prime_factors(p)
    for π, e in fac:
        a, b = π.polynomial().coefficients(sparse=False)[::-1]
        if b >= 0:
            return π
    return fac[0][0]

"""
Returns the p-valuation of x
"""
def val(p,x):
    v = 0
    while(x%p==0):
        x/=p
        v+=1
    return v

"""
Returns the prime factorization of b in the integers
"""
def int_fac(x):
    return Integer(x).factor()

"""
a: int 
b: int
Returns a cleaned (new_a, new_b), such that whenever a prime p divides new_a, p^3 doesn't divide new_b
"""
def clean(a,b):
    new_a = a
    new_b = b
    for p,exp in int_fac(b): 
        if a%p == 0 and exp >= 3:
            v1 = val(p,a)
            v2 = exp
            times = min(v1, v2 // 3)
            new_a /= p ** times
            new_b /= p**(3 * times)
    return new_a, new_b

"""
a: int 
b: list of lists denoting the prime factorization of b
Returns a cleaned (new_a, new_b_fac)
"""
def clean_fac(a, b_fac):
    new_b_fac = []
    new_b = 1
    for (p, exp) in b_fac:
        if exp < 3 or a % p != 0:
            new_b_fac.append([p, exp])
            new_b *= p ** exp
            continue
        v1 = val(p,a)
        times = min(v1, exp // 3)
        a /= p ** times
        new_exp = exp - (3 * times)
        if new_exp > 0:
            new_b_fac.append([p, new_exp])
            new_b *= p ** new_exp
        
    return a, new_b_fac, new_b

In [72]:
"""
As defined in Stephanie Chan's paper
"""
def delta(n):
    r = n %9
    if(r == 3 or r==6):
        return 1
    if (r== 4 or r ==5):
        return -1;
    return 0

"""
As defined in Stephanie Chan's paper
"""
def w2(n):
    v_2 = val(2,n)
    two_n =  2 * n
    if(v_2 == 2):
        two_n /= 8
    cnt = 0
    for p,exp in int_fac(two_n):
        if p%3 == 2:
            cnt+=1
    return cnt

"""
Returns the nullity of an l by m matrix mat
"""
def calc(l, m, mat):
    if l>0 :
        R = matrix(GF(3),np.array(mat))
        return m - R.rank()
    else:
        return m

def build_matrix(primes_to_check, primes_to_use, m, l, t, u1u2u3):
    R = np.empty((l, m), dtype=int) 
    for i in range(l): # l rows for the primes to check
        if i+1<=t:
            for j in range(m): # m columns for the primes to use
                if i==j: 
                    q_i = primes_to_use[i][0] ** primes_to_use[i][1]
                    res = cr(u1u2u3/q_i, prime_above(primes_to_check[i]))  
                    if res == 1: 
                        R[i][j] = 2 
                    elif res == 2: 
                        R[i][j] = 1 
                    else: 
                        R[i][j] = 0
                else:
                    R[i][j] = cr(primes_to_use[j][0]** primes_to_use[i][1], prime_above(primes_to_check[i]))
        else:
            for j in range(m): # m columns 
                R[i][j] = cr(primes_to_use[j][0], prime_above(primes_to_check[i]))
    return l,m,R

def solve(u1u2u3, c):
    dp = 27*(u1u2u3) - c**3
    primes_to_use = []
    t1 = []
    t2 = []
    primes_to_check = []
    for p,exp in Integer(u1u2u3).factor():
        if c%p == 0 and p%3 == 1:
            primes_to_check.append(p)
            t1.append([p,exp])
        else:
            t2.append([p,exp])
    primes_to_use = t1 + t2
        
    m = len(primes_to_use)
    t = len(primes_to_check)
    for p,exp in dp.factor():
        if p%3==1 and (u1u2u3)%p != 0:
            primes_to_check.append(p)
    l = len(primes_to_check)
    return build_matrix(primes_to_check, primes_to_use, m, l, t, u1u2u3)

def solve_fac(u1u2u3, u1u2u3_fac, c):
    p = 27*(u1u2u3) - c**3
    primes_to_use = []
    t1 = []
    t2 = []
    primes_to_check = []
    for p,exp in u1u2u3_fac:
        if c%p == 0 and p%3 == 1:
            primes_to_check.append(p)
            t1.append([p,exp])
        else:
            t2.append([p,exp])
    primes_to_use = t1 + t2
        
    m = len(primes_to_use)
    t = len(primes_to_check)
    for p,exp in dp.factor():
        if p%3==1 and (u1u2u3)%p != 0:
            primes_to_check.append(p)
    l = len(primes_to_check)
    return build_matrix(primes_to_check, primes_to_use, m, l, t, u1u2u3)

# WAB: y^2 + Axy + By = x^3
def constructWAB(A,B):
    c, u1u2u3 = clean(A,B)
    return solve(u1u2u3,c)


def constructWAB_fac(A,B_fac):
    c, u1u2u3_fac, u1u2u3 = clean_fac(A,B)
    return solve_fac(u1u2u3,u1u2u3_fac, c)


In [67]:
def computeByHeight(a, H):
    record = {}
    for B in range(a * H^3,(a+1) * H^3 + 1):
        if B%3 ==0: 
            continue
        for A in range(-(a+1) * H, - a * H + 1): 
            if A%3 ==0: 
                continue
            newA, newB = clean(A,B)
            if A != newA or B != newB: 
                continue
            l,m,mat = solve(newB,newA)
            mat = np.array(mat)
            if (l-m, m) not in record: 
                record[(l-m,m)] = {}
            dim = calc(l,m,mat)
            if dim not in record[(l-m,m)]:
                record[(l-m,m)][dim] = 1
            else:
                record[(l-m,m)][dim] += 1
        for A in range(a * H , (a+1) * H + 1): 
            if A%3 ==0: 
                continue
            newA, newB = clean(A,B)
            if A != newA or B != newB: 
                continue
            l,m,mat = solve(newB,newA)
            mat = np.array(mat)
            if (l-m, m) not in record: 
                record[(l-m,m)] = {}
            dim = calc(l,m,mat)
            if dim not in record[(l-m,m)]:
                record[(l-m,m)][dim] = 1
            else:
                record[(l-m,m)][dim] += 1
    return record

In [75]:
print("Starting Profiler...")

# 3. Setup the profiler
profiler = cProfile.Profile()
profiler.enable()  # Start timing

# --- RUN YOUR FUNCTION ---
computeByHeight(2, 7)
# -------------------------

profiler.disable() # Stop timing

# 4. Print results sorted by time
stats = pstats.Stats(profiler).sort_stats('tottime')
stats.print_stats(10) # Print only top 10 rows

Starting Profiler...
         1802004 function calls (1787768 primitive calls) in 4.813 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
    11079    0.380    0.000    0.849    0.000 {method 'linear_combination_of_rows' of 'sage.matrix.matrix0.Matrix' objects}
     3240    0.312    0.000    0.573    0.000 /home/dcz0711/yes/envs/sage/lib/python3.11/site-packages/sage/rings/number_field/number_field_ideal.py:1090(_cache_bnfisprincipal)
     3240    0.251    0.000    0.260    0.000 /home/dcz0711/yes/envs/sage/lib/python3.11/site-packages/sage/rings/number_field/number_field_ideal.py:589(<listcomp>)
36216/28830    0.249    0.000    1.125    0.000 /home/dcz0711/yes/envs/sage/lib/python3.11/site-packages/sage/modules/free_module.py:2179(_element_constructor_)
    24221    0.247    0.000    0.266    0.000 /home/dcz0711/yes/envs/sage/lib/python3.11/site-packages/sage/matrix/matrix_space.py:1038(_element_constructor_)
    10086    0.